In [25]:
import kagglehub
kaushikyh_indian_sign_language_words_with_landmarks_path = kagglehub.dataset_download('kaushikyh/indian-sign-language-words-with-landmarks')

print('Data source import complete.')


Data source import complete.


# Import Libraries

In [37]:
# Verify GPU and torch version (T4 works with cu128 natively - no reinstall needed)
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


Mon Apr  6 08:25:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import torch
torch.autograd.set_detect_anomaly(True)

print('pytorch version', torch.__version__)
print("GPU available:", torch.cuda.device_count())
#print('GPU name:',torch.cuda.get_device_name(0))

# Set the device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

pytorch version 2.10.0+cu128
GPU available: 2
cuda


In [7]:
# For Creating Dataloaders
from torch.utils.data import Dataset, DataLoader

In [8]:
# For Vision-ML model
import torchvision
from torchvision import transforms, models
from torchvision.transforms import v2
from torchvision.models import video as ptv

In [20]:
# For Deep Neural Network
import torch.nn.functional as F
from torch import nn
from torch.nn import Linear, Softmax, CrossEntropyLoss, ReLU, Flatten, Sequential

In [11]:
!pip install -q decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 89.1 MB/s eta 0:00:00:00:01:01


In [12]:
# Use PyTorch bridge for Decord
import decord

from decord.bridge import set_bridge
decord.bridge.set_bridge("torch")

from decord import VideoReader

In [13]:
import transformers

# For Tokenizers
from transformers import VivitImageProcessor

# For TPU
from transformers import set_seed
from torch.optim import AdamW
from accelerate import Accelerator, notebook_launcher

In [14]:
# For Display
from tqdm.notebook import tqdm
from torchsummary import summary

In [18]:
# General imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [8,5]

import random
import cv2
import os
import PIL
import gc
from glob import glob
import shutil

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Initialise Global Constants

In [26]:
output_dir = '/kaggle/working'
data_dir = kaushikyh_indian_sign_language_words_with_landmarks_path
image_dir = os.path.join(kaushikyh_indian_sign_language_words_with_landmarks_path, 'ProcessedData_vivit')

In [27]:
BATCH_SIZE = 32
print('BATCH_SIZE =',BATCH_SIZE)

IMAGE_PROCESSOR = 'google/vivit-b-16x2'
WEIGHTS = 'KINETICS400_V1'

CLIP_LENGTH = 16
print('Number of Frames =',CLIP_LENGTH)

CLIP_SIZE = 224
print('Image Dimension =', CLIP_SIZE,'X', CLIP_SIZE)

SEED = 42

BATCH_SIZE = 32
Number of Frames = 16
Image Dimension = 224 X 224


In [28]:
def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [29]:
seed_everything(SEED)

# Import metadata

In [30]:
classes = sorted(os.listdir(image_dir))
print(classes)

['afternoon', 'animal', 'bad', 'beautiful', 'big', 'bird', 'blind', 'cat', 'cheap', 'clothing', 'cold', 'cow', 'curved', 'deaf', 'dog', 'dress', 'dry', 'evening', 'expensive', 'famous', 'fast', 'female', 'fish', 'flat', 'friday', 'good', 'happy', 'hat', 'healthy', 'horse', 'hot', 'hour', 'light', 'long', 'loose', 'loud', 'minute', 'monday', 'month', 'morning', 'mouse', 'narrow', 'new', 'night', 'old', 'pant', 'pocket', 'quiet', 'sad', 'saturday', 'second', 'shirt', 'shoes', 'short', 'sick', 'skirt', 'slow', 'small', 'suit', 'sunday', 't_shirt', 'tall', 'thursday', 'time', 'today', 'tomorrow', 'tuesday', 'ugly', 'warm', 'wednesday', 'week', 'wet', 'wide', 'year', 'yesterday', 'young']


In [31]:
label_to_idx = {}
idx_to_label = {}

for idx, label in enumerate(classes):
  class_folder = os.path.join(image_dir, label)
  if os.path.exists(data_dir):
    label_to_idx[label] = idx
    idx_to_label[idx] = label

In [53]:
print(label_to_idx)

{'afternoon': 0, 'animal': 1, 'bad': 2, 'beautiful': 3, 'big': 4, 'bird': 5, 'blind': 6, 'cat': 7, 'cheap': 8, 'clothing': 9, 'cold': 10, 'cow': 11, 'curved': 12, 'deaf': 13, 'dog': 14, 'dress': 15, 'dry': 16, 'evening': 17, 'expensive': 18, 'famous': 19, 'fast': 20, 'female': 21, 'fish': 22, 'flat': 23, 'friday': 24, 'good': 25, 'happy': 26, 'hat': 27, 'healthy': 28, 'horse': 29, 'hot': 30, 'hour': 31, 'light': 32, 'long': 33, 'loose': 34, 'loud': 35, 'minute': 36, 'monday': 37, 'month': 38, 'morning': 39, 'mouse': 40, 'narrow': 41, 'new': 42, 'night': 43, 'old': 44, 'pant': 45, 'pocket': 46, 'quiet': 47, 'sad': 48, 'saturday': 49, 'second': 50, 'shirt': 51, 'shoes': 52, 'short': 53, 'sick': 54, 'skirt': 55, 'slow': 56, 'small': 57, 'suit': 58, 'sunday': 59, 't_shirt': 60, 'tall': 61, 'thursday': 62, 'time': 63, 'today': 64, 'tomorrow': 65, 'tuesday': 66, 'ugly': 67, 'warm': 68, 'wednesday': 69, 'week': 70, 'wet': 71, 'wide': 72, 'year': 73, 'yesterday': 74, 'young': 75}


In [32]:
print(idx_to_label)

{0: 'afternoon', 1: 'animal', 2: 'bad', 3: 'beautiful', 4: 'big', 5: 'bird', 6: 'blind', 7: 'cat', 8: 'cheap', 9: 'clothing', 10: 'cold', 11: 'cow', 12: 'curved', 13: 'deaf', 14: 'dog', 15: 'dress', 16: 'dry', 17: 'evening', 18: 'expensive', 19: 'famous', 20: 'fast', 21: 'female', 22: 'fish', 23: 'flat', 24: 'friday', 25: 'good', 26: 'happy', 27: 'hat', 28: 'healthy', 29: 'horse', 30: 'hot', 31: 'hour', 32: 'light', 33: 'long', 34: 'loose', 35: 'loud', 36: 'minute', 37: 'monday', 38: 'month', 39: 'morning', 40: 'mouse', 41: 'narrow', 42: 'new', 43: 'night', 44: 'old', 45: 'pant', 46: 'pocket', 47: 'quiet', 48: 'sad', 49: 'saturday', 50: 'second', 51: 'shirt', 52: 'shoes', 53: 'short', 54: 'sick', 55: 'skirt', 56: 'slow', 57: 'small', 58: 'suit', 59: 'sunday', 60: 't_shirt', 61: 'tall', 62: 'thursday', 63: 'time', 64: 'today', 65: 'tomorrow', 66: 'tuesday', 67: 'ugly', 68: 'warm', 69: 'wednesday', 70: 'week', 71: 'wet', 72: 'wide', 73: 'year', 74: 'yesterday', 75: 'young'}


In [55]:
# Collect all video files
video_path = []
labels_int = []
labels_text = []
for idx, label in enumerate(classes):
  class_folder = os.path.join(image_dir, label)
  #print(class_folder)
  video_file = glob(os.path.join(class_folder, '*.MOV'))
  #print(video_file)
  video_path.extend(video_file)
  labels_int.extend([idx] * len(video_file))
  labels_text.extend([label] * len(video_file))

In [56]:
print('input path size:', len(video_path))
print(video_path[:20])

input path size: 1166
['/kaggle/input/datasets/kaushikyh/indian-sign-language-words-with-landmarks/ProcessedData_vivit/afternoon/MVI_5512.MOV', '/kaggle/input/datasets/kaushikyh/indian-sign-language-words-with-landmarks/ProcessedData_vivit/afternoon/MVI_5064.MOV', '/kaggle/input/datasets/kaushikyh/indian-sign-language-words-with-landmarks/ProcessedData_vivit/afternoon/MVI_5511.MOV', '/kaggle/input/datasets/kaushikyh/indian-sign-language-words-with-landmarks/ProcessedData_vivit/afternoon/MVI_4656.MOV', '/kaggle/input/datasets/kaushikyh/indian-sign-language-words-with-landmarks/ProcessedData_vivit/afternoon/MVI_4657.MOV', '/kaggle/input/datasets/kaushikyh/indian-sign-language-words-with-landmarks/ProcessedData_vivit/afternoon/MVI_5513.MOV', '/kaggle/input/datasets/kaushikyh/indian-sign-language-words-with-landmarks/ProcessedData_vivit/afternoon/MVI_5514.MOV', '/kaggle/input/datasets/kaushikyh/indian-sign-language-words-with-landmarks/ProcessedData_vivit/afternoon/MVI_4655.MOV', '/kaggle/

In [57]:
print('labels size:', len(labels_int))
print(labels_int[:20])

labels size: 1166
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [58]:
print('labels size:', len(labels_text))
print(labels_text[:20])

labels size: 1166
['afternoon', 'afternoon', 'afternoon', 'afternoon', 'afternoon', 'afternoon', 'afternoon', 'afternoon', 'afternoon', 'afternoon', 'afternoon', 'animal', 'animal', 'animal', 'animal', 'animal', 'animal', 'animal', 'animal', 'animal']


# Train test split

In [59]:
train_path, val_path, train_label, val_label = train_test_split(video_path, labels_int, test_size=0.2, stratify=labels_text, random_state=SEED)
train_path, test_path, train_label, test_label = train_test_split(train_path, train_label, test_size=0.2, stratify=train_label, random_state=SEED)

In [60]:
print('Train data size:',len(train_path))
print('Val data size:',len(val_path))
print('Test data size:',len(test_path))

Train data size: 745
Val data size: 234
Test data size: 187


In [61]:
train_label[:5]

[54, 28, 25, 52, 5]

# Create Dataset

In [62]:
from transformers import VivitImageProcessor

image_processor = VivitImageProcessor(
    do_resize=True,
    size={"shortest_edge": 224},
    do_center_crop=True,
    crop_size={"height": 224, "width": 224},
    do_rescale=True,
    rescale_factor=1/255,
    do_normalize=True,
    image_mean=[0.5, 0.5, 0.5],
    image_std=[0.5, 0.5, 0.5],
)
print("Image processor created manually!")

Image processor created manually!


In [63]:
class CreateDataset(Dataset):
    def __init__(self, clip_length, image_processor, video_paths, labels, training=True):
        super().__init__()
        self.clip_length = clip_length
        self.image_processor = image_processor

        self.video_paths = video_paths
        self.labels = labels
        self.training = training

        # Initialize cache for decoded videos
        self.cache = {}

        # Pre-load and cache all videos with a progress bar
        print("Caching videos...")
        for idx, video_path in tqdm(enumerate(self.video_paths), total=len(self.video_paths), desc="Loading videos"):
            vr = VideoReader(video_path)
            video = vr.get_batch(list(range(len(vr))))
            self.cache[idx] = video[:self.clip_length]
            del vr
            gc.collect()
        print("Video caching complete.")

        # Define a transformation pipeline
        self.transform_train = v2.Compose([
                                    v2.ToImage(),
                                    v2.RandomPerspective(),
                                    v2.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.05),
                                    v2.ToDtype(torch.uint8, scale=False)
                                ])

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        video = self.cache[idx]

        if self.training:
            # Data Preperation for ML model with Augmentation
            video = self.transform_train(video.permute(0, 3, 1, 2))
        else:
            # Data Preperation for ML Model without Augmentation
            video = v2.functional.to_dtype(video.permute(0, 3, 1, 2), torch.uint8, scale=False)

        # Scaling the video to ML model's desired format
        video = self.image_processor(list(video), return_tensors='pt', input_data_format='channels_first')
        pixel_values = video['pixel_values'].squeeze(0)
        pixel_values = pixel_values.permute(1, 0, 2, 3)

        del video
        gc.collect()

        return {
                'labels': label,
                'pixel_values': pixel_values
                }

In [64]:
train_ds = CreateDataset(clip_length=CLIP_LENGTH,
                         image_processor=image_processor, training=True,
                         video_paths=train_path,
                         labels=train_label
                         )
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

Caching videos...


Loading videos:   0%|          | 0/745 [00:00<?, ?it/s]

Video caching complete.


In [65]:
val_ds = CreateDataset(clip_length=CLIP_LENGTH,
                         image_processor=image_processor, training=False,
                         video_paths=val_path,
                         labels=val_label
                         )
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE)

Caching videos...


Loading videos:   0%|          | 0/234 [00:00<?, ?it/s]

Video caching complete.


In [66]:
test_ds = CreateDataset(clip_length=CLIP_LENGTH,
                         image_processor=image_processor, training=True,
                         video_paths=test_path,
                         labels=test_label
                         )
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE)

Caching videos...


Loading videos:   0%|          | 0/187 [00:00<?, ?it/s]

Video caching complete.


# Testing custome image processing

In [67]:
# Convert images to numpy for visualization
def imgshow(img):
    img = img / 2 + 0.5  # Unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

In [68]:
inputs = next(iter(train_dl))
inp_pixel = inputs["pixel_values"]
inp_label = inputs["labels"]
print("inputs shape: ", inp_pixel.shape, inp_label.shape)
print("label", inp_label[0])
# Visualization skipped (numpy conflict with torch 2.2)


inputs shape:  torch.Size([32, 3, 16, 224, 224]) torch.Size([32])
label tensor(56)


In [69]:
inputs = next(iter(val_dl))
inp_pixel = inputs["pixel_values"]
inp_label = inputs["labels"]
print("inputs shape: ", inp_pixel.shape, inp_label.shape)
print("label", inp_label[0])
# Visualization skipped (numpy conflict with torch 2.2)


inputs shape:  torch.Size([32, 3, 16, 224, 224]) torch.Size([32])
label tensor(43)


In [70]:
inputs = next(iter(test_dl))
inp_pixel = inputs["pixel_values"]
inp_label = inputs["labels"]
print("inputs shape: ", inp_pixel.shape, inp_label.shape)
print("label", inp_label[0])
# Visualization skipped (numpy conflict with torch 2.2)


inputs shape:  torch.Size([32, 3, 16, 224, 224]) torch.Size([32])
label tensor(56)


# Testing model structure

In [71]:
#sample_model_1 = ptv.swin3d_s(weights=WEIGHTS).to(device)

In [72]:
#summary(sample_model_1, (3,32,224,224))

In [73]:
#sample_model_1

In [74]:
## Freeze all layers initially
#for param in sample_model_1.parameters():
#    param.requires_grad = False

In [75]:
#sample_model_1.features[6][1]

In [76]:
## Un-freeze only the last sequential layer
#for param in sample_model_1.features[6][1].parameters():
#    param.requires_grad = True

In [77]:
#summary(sample_model_1, (3,32,224,224))

# Model Building

In [78]:
hyperparameters = {
    "learning_rate": 0.0001,
    "num_epochs": 1000, # set to very high number
    "patience": 5, # early stopping
    "seed": SEED,
    "output_dir_pt": f"{output_dir}/swin_small_ISL_gpu_pt_1.pt",
    "attention_dropout": 0.2,
    "dropout_rate": 0.2,
    "num_classes": len(classes),
    # "norm_layer": nn.LayerNorm(normalized_shape=torch.Tensor([3])) # normalized_shape =. number of channels
}

In [79]:
from torch import nn
class SwinTClassifications(nn.Module):
    def __init__(self,classes, weights="DEFAULT"):
        super().__init__()
        self.classes = classes
        self.weights = weights
        # Load pre-trained model
        self.base_model = ptv.swin3d_s(weights=self.weights,
                                       #dropout=hyperparameters['dropout_rate'],
                                       #attention_dropout=hyperparameters['attention_dropout'],
                                       #norm_layer=hyperparameters['norm_layer'],
                                       )

        # Update the Classification layer to have 76 classes
        #self.base_model.head = torch.nn.Linear(self.base_model.head.in_features , len(self.classes))
        self.classification_head = nn.Sequential(torch.nn.Linear(self.base_model.head.in_features , len(self.classes)),
                                             #nn.Softmax(dim=1)
                                             )

        # Remove the classification head
        self.base_model.head = nn.Identity()

        ## Enable gradient checkpointing to save memory
        #self.base_model.gradient_checkpointing_enable()

        # Freeze all layers initially
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Un-Freeze the last Sequential layer
        for param in self.base_model.features[6].parameters():
            param.requires_grad = True
        # also unfreeze norm layer
        for param in self.base_model.norm.parameters():
            param.requires_grad = True

    def forward(self, x):
        x = self.base_model(x)  # Feature extraction
        x = self.classification_head(x)
        return x  # Return classes

In [80]:
def training_function():
    device = torch.device("cuda:0")
    print(f"Training on: {device}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    embedding_model = SwinTClassifications(classes=classes, weights=WEIGHTS).to(device)
    print(f"Model device: {next(embedding_model.parameters()).device}")

    set_seed(hyperparameters["seed"])

    criterion = torch.nn.CrossEntropyLoss().to(device)
    optimizer = AdamW(embedding_model.parameters(), lr=hyperparameters["learning_rate"])
    scaler = torch.cuda.amp.GradScaler()  # replaces fp16 from Accelerator

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=hyperparameters["patience"], min_lr=0.00001
    )

    epochs_no_improve = 0
    min_val_loss = float("inf")

    for epoch in range(hyperparameters["num_epochs"]):
        progress_bar = tqdm(range(len(train_dl)))
        progress_bar.set_description(f"Epoch: {epoch}")
        embedding_model.train()
        training_loss = []

        for batch in train_dl:
            pixel_value = batch['pixel_values'].to(device)
            label = batch['labels'].to(device)

            optimizer.zero_grad()

            # fp16 via GradScaler (same as Accelerator mixed_precision="fp16")
            with torch.cuda.amp.autocast():
                outputs = embedding_model(pixel_value)
                train_loss = criterion(outputs, label)

            scaler.scale(train_loss).backward()
            scaler.step(optimizer)
            scaler.update()

            del pixel_value, outputs, label
            gc.collect()

            training_loss.append(train_loss.detach().item())
            progress_bar.set_postfix({'loss': train_loss.detach().item()})
            progress_bar.update(1)

        training_loss_final = sum(training_loss) / len(training_loss)
        print(f"epoch {epoch}: learning rate: {scheduler.get_last_lr()}")
        print(f"epoch {epoch}: training loss: {training_loss_final}")

        embedding_model.eval()
        validation_loss = []
        for batch in val_dl:
            pixel_value = batch['pixel_values'].to(device)
            label = batch['labels'].to(device)

            with torch.no_grad(), torch.cuda.amp.autocast():
                outputs = embedding_model(pixel_value)
                val_loss = criterion(outputs, label)

            validation_loss.append(val_loss.item())

            del pixel_value, outputs, label
            gc.collect()

        validation_loss_final = sum(validation_loss) / len(validation_loss)
        print(f"epoch {epoch}: validation loss: {validation_loss_final}")
        print(f"GPU memory used: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")

        scheduler.step(validation_loss_final)

        if validation_loss_final < min_val_loss:
            epochs_no_improve = 0
            min_val_loss = validation_loss_final
            base, ext = os.path.splitext(hyperparameters['output_dir_pt'])
            checkpoint_path = f"{base}_epoch{epoch}_valloss{validation_loss_final:.4f}{ext}"
            torch.save(embedding_model.state_dict(), checkpoint_path)
            print(f"Checkpoint saved: {checkpoint_path}")
        else:
            epochs_no_improve += 1
            print(f"No improvement ({epochs_no_improve}/{hyperparameters['patience']})")
            if epochs_no_improve == hyperparameters["patience"]:
                print("Early stopping!")
                break

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["ACCELERATE_USE_CPU"] = "false"

# Verify accelerate sees GPU
from accelerate import Accelerator
test_acc = Accelerator(mixed_precision="fp16")
print(f"Accelerator device: {test_acc.device}")   # must say cuda:0
print(f"Num GPUs: {test_acc.num_processes}")
del test_acc

# Now run training
training_function()

Accelerator device: cuda
Num GPUs: 1
Training on: cuda:0
GPU: Tesla T4
Model device: cuda:0


/tmp/ipykernel_7178/2166369192.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # replaces fp16 from Accelerator


  0%|          | 0/24 [00:00<?, ?it/s]

/tmp/ipykernel_7178/2166369192.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


In [ ]:
import glob
import shutil
import os

# Find best checkpoint
checkpoints = glob.glob('/kaggle/working/swin_small_ISL_gpu_pt_1_epoch*_valloss*.pt')

if not checkpoints:
    print("No checkpoint found. Files in /kaggle/working:")
    for f in os.listdir('/kaggle/working'):
        print(f)
else:
    best_model_path = sorted(checkpoints)[-1]
    print(f"Best checkpoint: {best_model_path}")

    # ── Option 1: Download directly ───────────────────────────────────────────
    from IPython.display import FileLink, display
    display(FileLink(best_model_path))

    # ── Option 2: Save as Kaggle Dataset (persists forever) ───────────────────
    # Step 1: Create dataset folder
    dataset_dir = '/kaggle/working/isl_model_dataset'
    os.makedirs(dataset_dir, exist_ok=True)
    shutil.copy(best_model_path, dataset_dir)

    # Step 2: Create dataset metadata
    import json
    metadata = {
        "title": "ISL Swin3D Model",
        "id": "YOUR_KAGGLE_USERNAME/isl-swin3d-model",  # ← change username
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(f'{dataset_dir}/dataset-metadata.json', 'w') as f:
        json.dump(metadata, f)

    print(f"\nDataset folder ready: {dataset_dir}")
    print("Contents:")
    for f in os.listdir(dataset_dir):
        print(f" - {f}")

    # Step 3: Push to Kaggle as a dataset
    print("\nUploading to Kaggle...")
    result = os.popen(f'kaggle datasets create -p {dataset_dir} --dir-mode tar').read()
    print(result)
    print("Done! Find it at: https://www.kaggle.com/datasets/YOUR_KAGGLE_USERNAME/isl-swin3d-model")

# Load Model

In [33]:
class SwinTClassifications(nn.Module):
    def __init__(self,classes, weights="DEFAULT"):
        super().__init__()
        self.classes = classes
        self.weights = weights
        # Load pre-trained model
        self.base_model = ptv.swin3d_s(weights=self.weights)

        self.classification_head = nn.Sequential(torch.nn.Linear(self.base_model.head.in_features , len(self.classes)),
                                             #nn.Softmax(dim=1)
                                             )

        # Remove the classification head
        self.base_model.head = nn.Identity()

    def forward(self, x):
        x = self.base_model(x)  # Feature extraction
        x = self.classification_head(x)
        return x  # Return classes

In [34]:
import glob as glob_module

class SwinTClassifications(nn.Module):
    def __init__(self, classes, weights="DEFAULT"):
        super().__init__()
        self.classes = classes
        self.weights = weights
        self.base_model = ptv.swin3d_s(weights=self.weights)
        self.classification_head = nn.Sequential(
            torch.nn.Linear(self.base_model.head.in_features, len(self.classes))
        )
        self.base_model.head = nn.Identity()

    def forward(self, x):
        x = self.base_model(x)
        x = self.classification_head(x)
        return x

# Find best checkpoint
checkpoints = glob_module.glob("/kaggle/working/swin_small_ISL_gpu_pt_1_epoch*_valloss*.pt")
if not checkpoints:
    raise FileNotFoundError("No checkpoint found in /kaggle/working/")
best_ckpt = sorted(checkpoints)[-1]
print(f"Loading checkpoint: {best_ckpt}")

recon_model = SwinTClassifications(classes=classes, weights=WEIGHTS).to(device)
recon_model.load_state_dict(torch.load(best_ckpt, weights_only=True, map_location=device))
recon_model.eval()
print("Model loaded successfully!")


Loading checkpoint: /kaggle/working/swin_small_ISL_gpu_pt_1_epoch9_valloss1.3639.pt
Downloading: "https://download.pytorch.org/models/swin3d_s-da41c237.pth" to /root/.cache/torch/hub/checkpoints/swin3d_s-da41c237.pth


100%|██████████| 218M/218M [00:01<00:00, 187MB/s]  


Model loaded successfully!


# Make Predictions

# Analyze predictions

In [197]:
from sklearn.metrics import classification_report, accuracy_score

def testing_function(model):
    model.eval()
    test_labels_list = []
    test_preds_list = []
    test_outputs_list = []

    with torch.no_grad():
        for batch in tqdm(test_dl, desc="Testing"):
            pixel_value = batch['pixel_values'].to(device)
            label = batch['labels'].to(device)

            with torch.cuda.amp.autocast():
                outputs = model(pixel_value)

            softmax = torch.nn.functional.softmax(outputs, dim=-1)
            preds = softmax.argmax(-1)

            test_preds_list.append(preds.cpu())
            test_outputs_list.append(outputs.cpu())
            test_labels_list.append(label.cpu())

            del pixel_value, outputs, label
            gc.collect()

    actual_labels      = torch.cat(test_labels_list, 0)
    predicted_labels   = torch.cat(test_preds_list, 0)
    predicted_outputs  = torch.cat(test_outputs_list, 0)

    return actual_labels, predicted_labels, predicted_outputs


# Run testing
actual_labels, predicted_labels, predicted_outputs = testing_function(recon_model)

# Results
acc = accuracy_score(actual_labels.numpy(), predicted_labels.numpy())
print(f"\n{'='*40}")
print(f"Top-1 Accuracy: {acc*100:.2f}%")
print(f"{'='*40}\n")
print(classification_report(
    actual_labels.numpy(),
    predicted_labels.numpy(),
    target_names=classes,
    digits=3
))

Testing:   0%|          | 0/6 [00:00<?, ?it/s]

/tmp/ipykernel_7178/2299729881.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Top-1 Accuracy: 66.84%

              precision    recall  f1-score   support

   afternoon      0.667     1.000     0.800         2
      animal      1.000     0.667     0.800         3
         bad      0.750     0.750     0.750         4
   beautiful      0.333     1.000     0.500         1
         big      1.000     0.500     0.667         4
        bird      1.000     1.000     1.000         3
       blind      0.500     1.000     0.667         1
         cat      0.667     0.667     0.667         3
       cheap      0.500     1.000     0.667         1
    clothing      0.750     1.000     0.857         3
        cold      0.750     1.000     0.857         3
         cow      1.000     1.000     1.000         3
      curved      1.000     1.000     1.000         1
        deaf      0.000     0.000     0.000         1
         dog      0.400     0.667     0.500         3
       dress      0.500     0.333     0.400         3
         dry      0.333     0.333     0.333         3
  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
!pip install huggingface_hub -q

from huggingface_hub import HfApi, login

# Login with your token
login(token="hf_YOUR_TOKEN")

api = HfApi()

# Create repo
api.create_repo(
    repo_id="USERNAME/isl-swin3d-model",
    repo_type="model",
    exist_ok=True
)

# Upload best checkpoint
api.upload_file(
    path_or_fileobj="/kaggle/working/ISL_best_model.pt",
    path_in_repo="ISL_best_model.pt",
    repo_id="USERNAME/isl-swin3d-model",
    repo_type="model"
)

print("Model uploaded to: https://huggingface.co/USERNAME/isl-swin3d-model")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Model uploaded to: https://huggingface.co/Creator-090/isl-swin3d-model


In [35]:
from torchsummary import summary

# Ensure the model is on the correct device (GPU/CPU)
recon_model.to(device)

# Provide the input size based on your constants (Channels, Time/Frames, Height, Width)
# Based on your CLIP_LENGTH = 16 and CLIP_SIZE = 224
summary(recon_model, (3, 16, 224, 224))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv3d-1        [-1, 96, 8, 56, 56]           9,312
         LayerNorm-2        [-1, 8, 56, 56, 96]             192
      PatchEmbed3d-3        [-1, 8, 56, 56, 96]               0
           Dropout-4        [-1, 8, 56, 56, 96]               0
         LayerNorm-5        [-1, 8, 56, 56, 96]             192
ShiftedWindowAttention3d-6        [-1, 8, 56, 56, 96]               0
   StochasticDepth-7        [-1, 8, 56, 56, 96]               0
         LayerNorm-8        [-1, 8, 56, 56, 96]             192
            Linear-9       [-1, 8, 56, 56, 384]          37,248
             GELU-10       [-1, 8, 56, 56, 384]               0
          Dropout-11       [-1, 8, 56, 56, 384]               0
           Linear-12        [-1, 8, 56, 56, 96]          36,960
          Dropout-13        [-1, 8, 56, 56, 96]               0
  StochasticDepth-14        [-1, 